## IMPORTS AND INITIAL SETUP

In [1]:
# Core TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator

2025-09-11 10:20:21.880789: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-11 10:20:21.880852: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-11 10:20:21.882996: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-11 10:20:21.893155: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Data manipulation and analysis
import numpy as np
import pandas as pd

In [3]:
# Machine learning utilities
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score
)

In [4]:
# Visualization
import matplotlib.pyplot as plt
plt.style.use("default")

In [5]:
# System and utility imports
import os
import shutil
import datetime
import random

In [6]:
# Install required packages FIRST
print("Installing required packages...")
!pip install opencv-python-headless --quiet
!pip install openpyxl --quiet

Installing required packages...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.6 which is incompatible.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.15.0 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 23.3.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [7]:
# Now import cv2 after installation
import cv2
print(f"OpenCV version: {cv2.__version__}")

OpenCV version: 4.12.0


In [8]:
# Experiment tracking imports
from comet_ml import Experiment
from dotenv import load_dotenv

## GPU CONFIGURATION AND VERIFICATION

In [9]:
# List all available devices
print("\n" + "="*50)
print("DEVICE CONFIGURATION")
print("="*50)

devices = tf.config.list_physical_devices()
print("Available devices:", devices)


DEVICE CONFIGURATION
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [10]:
# Check for GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Found {len(gpus)} GPU(s):")
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu}")
    
    # Configure GPU memory growth to avoid OOM
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth configured successfully")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("WARNING: No GPU found. Training will use CPU.")

Found 1 GPU(s):
  GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU memory growth configured successfully


In [11]:
# Verify GPU device name
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    print(f"WARNING: GPU device not found at expected location. Found: {device_name}")
else:
    print(f"GPU confirmed at: {device_name}")

GPU confirmed at: /device:GPU:0


2025-09-11 10:20:35.049712: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 79196 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:b7:00.0, compute capability: 8.0


In [12]:
# Set default device context
tf.device('/device:GPU:0')

print("="*50)
print("SETUP COMPLETE")
print("="*50)

SETUP COMPLETE


2025-09-11 10:20:35.061881: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79196 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:b7:00.0, compute capability: 8.0


## COMET ML EXPERIMENT SETUP AND IMAGE DATA LOADING

In [13]:
# Initialize Comet ML experiment for comprehensive tracking
experiment = Experiment(
    api_key=os.getenv("COMET_API_KEY"),         
    project_name="multi_modal_development",   # Updated project name
    workspace=None,  # Will use default workspace
    auto_histogram_weight_logging=True,         
    auto_histogram_gradient_logging=True,       
    auto_histogram_activation_logging=True,
    auto_log_co2=True,  # Track carbon footprint
    log_code=True,      # Log the code for reproducibility
    log_graph=True      # Log model architecture
)

print(f"Comet experiment initialized: {experiment.get_key()}")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, keras, sklearn, tensorflow.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Couldn't find a Git repository in '/app' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: Experiment is live on comet.com https://www.comet.com/anabiosi-data/multi-modal-development/17342727d57d4033ba7838fe50f10d21



Comet experiment initialized: 17342727d57d4033ba7838fe50f10d21


### DATA ARTIFACT RETRIEVAL AND LOCAL SETUP

In [14]:
# Retrieve the dataset artifact from Comet ML
try:
    logged_artifact = experiment.get_artifact("elastography_images_merged")
    print("Successfully retrieved artifact: elastography_images_merged")
    
    # Check if images are already downloaded locally
    local_image_path = "./Elastography_images"
    if not os.path.exists(local_image_path):
        print("Downloading images to local directory...")
        local_artifact = logged_artifact.download("./")
        print(f"Images downloaded to: {os.path.abspath(local_image_path)}")
    else:
        print(f"Images already exist at: {os.path.abspath(local_image_path)}")
        
except Exception as e:
    print(f"Error retrieving artifact: {e}")
    print("Please ensure the artifact 'elastography_images_merged' exists in your Comet project")
    raise

Successfully retrieved artifact: elastography_images_merged


COMET INFO: Artifact 'anabiosi-data/elastography_images_merged:6.0.0' download has been started asynchronously
COMET INFO: Still downloading 1598 file(s), remaining 722.63 MB/722.63 MB
COMET INFO: Still downloading 1597 file(s), remaining 722.17 MB/722.63 MB, Throughput 31.24 KB/s, ETA ~23675s
COMET INFO: Artifact 'anabiosi-data/elastography_images_merged:6.0.0' has been successfully downloaded


Images downloaded to: /app/Elastography_images


### IMAGE DATA LOADING AND PREPROCESSING

In [15]:
# Set reproducibility seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

In [16]:
# Extract class names from artifact metadata
class_names = logged_artifact.metadata["classes"]
print(f"Class names: {class_names}")

Class names: ['response', 'stable', 'non-response']


In [17]:
# Configure image data generator
IMG_HEIGHT, IMG_WIDTH = 300, 400
BATCH_SIZE_LOAD = 1579  # Load all images at once

dataset_generator = ImageDataGenerator()
dataset = dataset_generator.flow_from_directory(
    local_image_path,
    batch_size=BATCH_SIZE_LOAD,
    class_mode='sparse',              # Integer labels (0, 1, 2)
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    shuffle=False,                    # Keep original order for alignment
    classes=class_names,              # Explicit class order
    interpolation='bilinear'          # Better quality resizing
)

print(f"Dataset configuration:")
print(f"  - Target size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"  - Batch size: {BATCH_SIZE_LOAD}")
print(f"  - Classes: {dataset.class_indices}")
print(f"  - Total samples found: {dataset.samples}")

Found 1578 images belonging to 3 classes.
Dataset configuration:
  - Target size: 300x400
  - Batch size: 1579
  - Classes: {'response': 0, 'stable': 1, 'non-response': 2}
  - Total samples found: 1578


In [18]:
# Load all images and labels into memory
print("\nLoading all images into memory...")
with tf.device('/device:GPU:0'):
    x_images, y_labels = dataset.next()

print(f"Successfully loaded:")
print(f"  - Images shape: {x_images.shape}")
print(f"  - Labels shape: {y_labels.shape}")
print(f"  - Image dtype: {x_images.dtype}")
print(f"  - Label dtype: {y_labels.dtype}")


Loading all images into memory...
Successfully loaded:
  - Images shape: (1578, 300, 400, 3)
  - Labels shape: (1578,)
  - Image dtype: float32
  - Label dtype: float32


In [19]:
# Verify class distribution
label_counts = np.bincount(y_labels.astype(int))
print(f"\nClass distribution:")
for i, (class_name, count) in enumerate(zip(class_names, label_counts)):
    print(f"  - {class_name}: {count} samples ({count/len(y_labels)*100:.1f}%)")


Class distribution:
  - response: 573 samples (36.3%)
  - stable: 492 samples (31.2%)
  - non-response: 513 samples (32.5%)


In [20]:
# Log dataset info to Comet
experiment.log_parameter("dataset_total_samples", len(y_labels))
experiment.log_parameter("image_height", IMG_HEIGHT)
experiment.log_parameter("image_width", IMG_WIDTH)
experiment.log_parameter("num_classes", len(class_names))
for i, (class_name, count) in enumerate(zip(class_names, label_counts)):
    experiment.log_parameter(f"class_{i}_{class_name}_count", count)

## EXCEL DATA LOADING AND ALIGNMENT WITH IMAGES

In [21]:
# Load the Excel file containing elastic modulus data
excel_path = './clustering_all_v5.xlsx'

try:
    df_excel = pd.read_excel(excel_path)
    print(f"Excel file loaded successfully: {excel_path}")
    print(f"Shape: {df_excel.shape}")
    print(f"Columns: {list(df_excel.columns)}")
    
except FileNotFoundError:
    print(f"ERROR: Excel file not found at {excel_path}")
    print("Please ensure the file exists in the current directory")
    raise
except Exception as e:
    print(f"ERROR loading Excel file: {e}")
    raise

# Display first few rows for verification
print(f"\nFirst 5 rows of Excel data:")
print(df_excel.head())

Excel file loaded successfully: ./clustering_all_v5.xlsx
Shape: (1578, 25)
Columns: ['name', 'Respone/stable/non-Response', 'Elastic Modulus SWE (kPa)', 'Perfused Area ', 'Cell lines', 'Type of Cancer', 'Therapy', 'final dimensions  x', 'final dimensions  y', 'final dimensions  z', 'Final Volume', 'initial dimensions  x', 'Initial dimensions  y', 'Initial dimensions  z', 'Initial Volume', 'relative volume', 'stress kPa from comsol ', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24']

First 5 rows of Excel data:
                                         name  Respone/stable/non-Response  \
0  4T1 LOOK DAY29_1C1_1A_b_cropped_square.tif                            1   
1  4T1 LOOK DAY29_1C1_1A_c_cropped_square.tif                            1   
2    4T1 LOOK DAY29_1C1_1A_cropped_square.tif                            1   
3  4T1 LOOK DAY29_1C1_1D_c_cropped_square.tif                            1   
4    4T1 LOOK DAY29_1C1

### DATA CLEANING AND PREPARATION

In [22]:
# Extract only the columns we need
required_columns = ['name', 'Elastic Modulus SWE (kPa)']
missing_cols = [col for col in required_columns if col not in df_excel.columns]

if missing_cols:
    print(f"ERROR: Missing required columns: {missing_cols}")
    print(f"Available columns: {list(df_excel.columns)}")
    raise KeyError(f"Required columns not found: {missing_cols}")

df_clean = df_excel[required_columns].copy()

# Rename columns for easier handling
df_clean.rename(columns={
    'name': 'filename',
    'Elastic Modulus SWE (kPa)': 'elastic_modulus_kpa'
}, inplace=True)

# Fix Unicode character issue in filenames (Τ → T)
df_clean['filename'] = df_clean['filename'].str.replace('\u03A4', 'T', regex=False)

# Remove any duplicate filenames
initial_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['filename'])
final_count = len(df_clean)

if initial_count != final_count:
    print(f"WARNING: Removed {initial_count - final_count} duplicate filename entries")

print(f"Cleaned Excel data:")
print(f"  - Shape: {df_clean.shape}")
print(f"  - Filename column: {df_clean['filename'].dtype}")
print(f"  - Elastic modulus column: {df_clean['elastic_modulus_kpa'].dtype}")

# Check for missing values
missing_filenames = df_clean['filename'].isnull().sum()
missing_kpa = df_clean['elastic_modulus_kpa'].isnull().sum()

if missing_filenames > 0:
    print(f"WARNING: {missing_filenames} missing filename entries")
if missing_kpa > 0:
    print(f"WARNING: {missing_kpa} missing elastic modulus entries")

print(f"\nElastic modulus statistics:")
print(df_clean['elastic_modulus_kpa'].describe())

Cleaned Excel data:
  - Shape: (1578, 2)
  - Filename column: object
  - Elastic modulus column: float64

Elastic modulus statistics:
count    1578.000000
mean       38.445133
std        12.672226
min        12.500000
25%        27.350000
50%        37.106151
75%        48.408640
max        63.485778
Name: elastic_modulus_kpa, dtype: float64


## IMAGE-EXCEL DATA ALIGNMENT

In [23]:
# Get the filenames from the image dataset
image_filenames = dataset.filenames  # Full paths like 'class/filename.jpg'
image_basenames = [os.path.basename(path) for path in image_filenames]

print(f"Image dataset info:")
print(f"  - Total image files: {len(image_filenames)}")
print(f"  - Sample filenames: {image_basenames[:5]}")

# Create filename lookup dictionary from Excel
excel_filenames_set = set(df_clean['filename'].values)
filename_to_kpa = dict(zip(df_clean['filename'], df_clean['elastic_modulus_kpa']))

print(f"\nExcel data info:")
print(f"  - Unique filenames in Excel: {len(excel_filenames_set)}")
print(f"  - Sample Excel filenames: {list(excel_filenames_set)[:5]}")

Image dataset info:
  - Total image files: 1578
  - Sample filenames: ['4T1 LOOK DAY29_1C1_1A_b_cropped_square.tif', '4T1 LOOK DAY29_1C1_1A_c_cropped_square.tif', '4T1 LOOK DAY29_1C1_1A_cropped_square.tif', '4T1 LOOK DAY29_1C1_1D_c_cropped_square.tif', '4T1 LOOK DAY29_1C1_1D_cropped_square.tif']

Excel data info:
  - Unique filenames in Excel: 1578
  - Sample Excel filenames: ['4T1_9_FM_1S.tif', '4T1-BOSENTAN_IMM_-IMMUNOC5_1A_d_cropped_square.tif', 'E0771_FM_8_2NR.tif', '4T1-BOSENTAN 0.2-CAGE32A_e_cropped_square.tif', 'B6-CONTROL-_B_-CAGE11D1A_cropped_square.tif']


In [24]:
# Find matches and mismatches
image_basenames_set = set(image_basenames)
matched_files = image_basenames_set.intersection(excel_filenames_set)
missing_in_excel = image_basenames_set - excel_filenames_set
missing_in_images = excel_filenames_set - image_basenames_set

print(f"\nAlignment analysis:")
print(f"  - Files in both image and Excel: {len(matched_files)}")
print(f"  - Files in images but not Excel: {len(missing_in_excel)}")
print(f"  - Files in Excel but not images: {len(missing_in_images)}")

if missing_in_excel:
    print(f"  - Examples missing in Excel: {list(missing_in_excel)[:10]}")
if missing_in_images:
    print(f"  - Examples missing in images: {list(missing_in_images)[:10]}")


Alignment analysis:
  - Files in both image and Excel: 1578
  - Files in images but not Excel: 0
  - Files in Excel but not images: 0


In [25]:
# Create aligned numeric array
print(f"\nCreating aligned numeric features...")
X_numeric = np.array([filename_to_kpa[basename] for basename in image_basenames], 
                     dtype=np.float32).reshape(-1, 1)

print(f"Numeric features created:")
print(f"  - Shape: {X_numeric.shape}")
print(f"  - Data type: {X_numeric.dtype}")
print(f"  - Min value: {X_numeric.min():.3f}")
print(f"  - Max value: {X_numeric.max():.3f}")
print(f"  - Mean value: {X_numeric.mean():.3f}")


Creating aligned numeric features...
Numeric features created:
  - Shape: (1578, 1)
  - Data type: float32
  - Min value: 12.500
  - Max value: 63.486
  - Mean value: 38.445


### DATA INTEGRITY VERIFICATION

In [26]:
# Verify all arrays have same length
assert x_images.shape[0] == X_numeric.shape[0] == y_labels.shape[0], \
    f"Shape mismatch: images={x_images.shape[0]}, numeric={X_numeric.shape[0]}, labels={y_labels.shape[0]}"

print(f"✓ All data arrays properly aligned:")
print(f"  - Images: {x_images.shape}")
print(f"  - Numeric: {X_numeric.shape}")
print(f"  - Labels: {y_labels.shape}")

✓ All data arrays properly aligned:
  - Images: (1578, 300, 400, 3)
  - Numeric: (1578, 1)
  - Labels: (1578,)


In [27]:
# Spot check random samples
print(f"\nRandom sample verification:")
verification_indices = np.random.choice(len(image_basenames), size=5, replace=False)

for idx in verification_indices:
    filename = image_basenames[idx]
    kpa_aligned = X_numeric[idx, 0]
    kpa_excel = filename_to_kpa[filename]
    label = y_labels[idx]
    class_name = class_names[int(label)]
    
    print(f"  Sample {idx:3d}: {filename}")
    print(f"    Label: {label} ({class_name})")
    print(f"    kPa aligned: {kpa_aligned:.4f}")
    print(f"    kPa Excel: {kpa_excel:.4f}")
    print(f"    Match: {np.isclose(kpa_aligned, kpa_excel)}")


Random sample verification:
  Sample 460: E0771_FM_6_0R.tif
    Label: 0.0 (response)
    kPa aligned: 33.5000
    kPa Excel: 33.5000
    Match: True
  Sample 1458: E0771_TRAN_MIC_DAY11C13_1D_b_cropped_square.tif
    Label: 2.0 (non-response)
    kPa aligned: 61.3995
    kPa Excel: 61.3995
    Match: True
  Sample 1428: E0771_17_M2NR.tif
    Label: 2.0 (non-response)
    kPa aligned: 38.5000
    kPa Excel: 38.5000
    Match: True
  Sample 1278: B6-BOSENTAN-10-CAGE91D_cropped_square.tif
    Label: 2.0 (non-response)
    kPa aligned: 49.7829
    kPa Excel: 49.7829
    Match: True
  Sample 116: 4T1-BOSENTAN 5-CAGE81A_cropped_square.tif
    Label: 0.0 (response)
    kPa aligned: 20.5907
    kPa Excel: 20.5907
    Match: True


In [28]:
# Log alignment statistics to Comet
experiment.log_parameter("excel_total_entries", len(df_clean))
experiment.log_parameter("matched_files", len(matched_files))
experiment.log_parameter("missing_in_excel", len(missing_in_excel))
experiment.log_parameter("missing_in_images", len(missing_in_images))
experiment.log_parameter("kpa_min", float(X_numeric.min()))
experiment.log_parameter("kpa_max", float(X_numeric.max()))
experiment.log_parameter("kpa_mean", float(X_numeric.mean()))
experiment.log_parameter("kpa_std", float(X_numeric.std()))

## TRAIN/VALIDATION/TEST SPLIT AND PREPROCESSING

### INITIAL DATA SHUFFLE

In [29]:
print("Shuffling all data with fixed seed for reproducibility...")

# Shuffle all data together to maintain alignment
X_img_all, X_num_all, y_all = shuffle(
    x_images,     # Image array
    X_numeric,    # Numeric kPa array  
    y_labels,     # Class labels
    random_state=SEED
)

print(f"Data shuffled:")
print(f"  - Images: {X_img_all.shape}")
print(f"  - Numeric: {X_num_all.shape}")
print(f"  - Labels: {y_all.shape}")

Shuffling all data with fixed seed for reproducibility...
Data shuffled:
  - Images: (1578, 300, 400, 3)
  - Numeric: (1578, 1)
  - Labels: (1578,)


### STRATIFIED TRAIN/VALIDATION/TEST SPLIT

In [30]:
# Complete 5-Fold Cross-Validation 
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

print(f"Starting {N_FOLDS}-fold cross-validation with {len(y_all)} total samples")

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_img_all, y_all), 1):
    
    print(f"\n{'='*20} FOLD {fold_num}/{N_FOLDS} {'='*20}")
    
    # Split data for this fold
    X_img_train, X_img_test = X_img_all[train_idx], X_img_all[test_idx]
    X_num_train, X_num_test = X_num_all[train_idx], X_num_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]
    
    # Create validation split (20% of training data)
    val_size = int(0.2 * len(X_img_train))
    X_img_val = X_img_train[-val_size:]
    X_img_train = X_img_train[:-val_size]
    X_num_val = X_num_train[-val_size:]
    X_num_train = X_num_train[:-val_size]
    y_val = y_train[-val_size:]
    y_train = y_train[:-val_size]
    
    print(f"Fold {fold_num} data sizes:")
    print(f"  Train: {len(y_train)} samples")
    print(f"  Validation: {len(y_val)} samples") 
    print(f"  Test: {len(y_test)} samples")
    
    # Scale numeric features for this fold
    scaler = StandardScaler()
    X_num_train_scaled = scaler.fit_transform(X_num_train)
    X_num_val_scaled = scaler.transform(X_num_val)
    X_num_test_scaled = scaler.transform(X_num_test)
    
    # Dataset creation - THIS IS INSIDE THE LOOP
    BATCH_SIZE = 32
    AUTOTUNE = tf.data.AUTOTUNE
    
    def preprocess_images(image, numeric, label):
        image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.image.adjust_contrast(image, 0.5)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return (image, numeric), label
    
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.05),
    ], name="conservative_swe_augmentation")
    
    def apply_augmentation(image, numeric, label):
        augmented_image = data_augmentation(image)
        return augmented_image, numeric, label
    
    def create_tf_dataset(images, numerics, labels, shuffle=False, augment_data=False):
        with tf.device('/CPU:0'):
            dataset = tf.data.Dataset.from_tensor_slices((images, numerics, labels))
        if shuffle:
            dataset = dataset.shuffle(buffer_size=len(images), seed=SEED)
        if augment_data:
            dataset = dataset.map(apply_augmentation, num_parallel_calls=AUTOTUNE)
        dataset = dataset.map(preprocess_images, num_parallel_calls=AUTOTUNE)
        return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    
    # Create datasets for this fold
    train_dataset = create_tf_dataset(X_img_train, X_num_train_scaled, y_train, shuffle=True, augment_data=True)
    val_dataset = create_tf_dataset(X_img_val, X_num_val_scaled, y_val, shuffle=False, augment_data=False)
    test_dataset = create_tf_dataset(X_img_test, X_num_test_scaled, y_test, shuffle=False, augment_data=False)
    
    print(f"Datasets created for fold {fold_num}")
    print(f"Fold {fold_num} setup complete - ready for model training")

print("\n5-fold cross-validation setup complete")

Starting 5-fold cross-validation with 1578 total samples

==================== FOLD 1/5 ====================
Fold 1 data sizes:
  Train: 1010 samples
  Validation: 252 samples
  Test: 316 samples
Datasets created for fold 1
Fold 1 setup complete - ready for model training

==================== FOLD 2/5 ====================
Fold 2 data sizes:
  Train: 1010 samples
  Validation: 252 samples
  Test: 316 samples
Datasets created for fold 2
Fold 2 setup complete - ready for model training

==================== FOLD 3/5 ====================
Fold 3 data sizes:
  Train: 1010 samples
  Validation: 252 samples
  Test: 316 samples
Datasets created for fold 3
Fold 3 setup complete - ready for model training

==================== FOLD 4/5 ====================
Fold 4 data sizes:
  Train: 1011 samples
  Validation: 252 samples
  Test: 315 samples
Datasets created for fold 4
Fold 4 setup complete - ready for model training

==================== FOLD 5/5 ====================
Fold 5 data sizes:
  Train:

### NUMERIC FEATURE STANDARDIZATION

We standardize the numeric features for several important reasons:
1. Scale Differences
Your elastic modulus values are likely in a range like 20-200 kPa, while your image features (after CNN processing) might be in ranges like 0-1 or -2 to +2. Without standardization, the model would be biased toward the larger-scale features.

2. Neural Network Training Stability
Neural networks train much better when all inputs have similar scales. Large differences in input scales can cause:
Gradient instability (exploding/vanishing gradients)
Slower convergence
Poor weight initialization effectiveness
Difficulty for the optimizer to find good solutions

3. Feature Importance Balance
In your multimodal model, you want both the image features and elastic modulus to contribute meaningfully. If elastic modulus values are much larger in magnitude, the model might ignore the image features entirely.

4. Data Leakage Prevention
The critical part is fitting the scaler ONLY on training data. If we used statistics from the entire dataset (including validation/test), we'd be "leaking" information about future data into our training process. This would make our validation/test performance artificially optimistic.

5. Transformer/Attention Mechanisms
These architectures are particularly sensitive to input scales because they use dot-product attention. Large input values can push the softmax function into saturation regions where gradients become very small.
Think of it this way: if one patient has an elastic modulus of 150 kPa and another has 25 kPa, but after CNN processing their image features are both around 0.5, the model will heavily weight the numeric difference and potentially ignore subtle but important image differences.

Standardization puts everything on a level playing field so the model can learn the optimal combination of both modalities.

In [31]:
# Complete 5-Fold Cross-Validation with Model Training
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

print(f"Starting {N_FOLDS}-fold cross-validation with {len(y_all)} total samples")

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_img_all, y_all), 1):
    
    print(f"\n{'='*20} FOLD {fold_num}/{N_FOLDS} {'='*20}")
    
    # Split data for this fold
    X_img_train, X_img_test = X_img_all[train_idx], X_img_all[test_idx]
    X_num_train, X_num_test = X_num_all[train_idx], X_num_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]
    
    # Create validation split (20% of training data)
    val_size = int(0.2 * len(X_img_train))
    X_img_val = X_img_train[-val_size:]
    X_img_train = X_img_train[:-val_size]
    X_num_val = X_num_train[-val_size:]
    X_num_train = X_num_train[:-val_size]
    y_val = y_train[-val_size:]
    y_train = y_train[:-val_size]
    
    print(f"Fold {fold_num} data sizes:")
    print(f"  Train: {len(y_train)} samples")
    print(f"  Validation: {len(y_val)} samples") 
    print(f"  Test: {len(y_test)} samples")
    
    # Scale numeric features for this fold
    scaler = StandardScaler()
    X_num_train_scaled = scaler.fit_transform(X_num_train)
    X_num_val_scaled = scaler.transform(X_num_val)
    X_num_test_scaled = scaler.transform(X_num_test)
    
    # Dataset creation
    BATCH_SIZE = 32
    AUTOTUNE = tf.data.AUTOTUNE
    
    def preprocess_images(image, numeric, label):
        image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.image.adjust_contrast(image, 0.5)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return (image, numeric), label
    
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.05),
    ], name="conservative_swe_augmentation")
    
    def apply_augmentation(image, numeric, label):
        augmented_image = data_augmentation(image)
        return augmented_image, numeric, label
    
    def create_tf_dataset(images, numerics, labels, shuffle=False, augment_data=False):
        with tf.device('/CPU:0'):
            dataset = tf.data.Dataset.from_tensor_slices((images, numerics, labels))
        if shuffle:
            dataset = dataset.shuffle(buffer_size=len(images), seed=SEED)
        if augment_data:
            dataset = dataset.map(apply_augmentation, num_parallel_calls=AUTOTUNE)
        dataset = dataset.map(preprocess_images, num_parallel_calls=AUTOTUNE)
        return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    
    # Create datasets for this fold
    train_dataset = create_tf_dataset(X_img_train, X_num_train_scaled, y_train, shuffle=True, augment_data=True)
    val_dataset = create_tf_dataset(X_img_val, X_num_val_scaled, y_val, shuffle=False, augment_data=False)
    test_dataset = create_tf_dataset(X_img_test, X_num_test_scaled, y_test, shuffle=False, augment_data=False)
    
    print(f"Datasets created for fold {fold_num}")
    
    # Model architecture
    @tf.keras.utils.register_keras_serializable()
    class ParametrisedCompatibility(layers.Layer):
        def __init__(self, kernel_regularizer=None, **kwargs):
            super().__init__(**kwargs)
            self.kernel_regularizer = regularizers.get(kernel_regularizer)
            
        def build(self, input_shape):
            C = input_shape[0][-1]
            self.u = self.add_weight(
                name='u', shape=(C, 1), initializer='uniform',
                regularizer=self.kernel_regularizer, trainable=True
            )
            super().build(input_shape)
            
        def call(self, inputs):
            local, g = inputs
            g_proj = K.expand_dims(K.expand_dims(g, 1), 1)
            combined = tf.nn.tanh(local + g_proj)
            return K.dot(combined, self.u)
            
        def compute_output_shape(self, input_shape):
            return (input_shape[0][0], input_shape[0][1], input_shape[0][2], 1)
            
        def get_config(self):
            config = super().get_config()
            config.update({"kernel_regularizer": regularizers.serialize(self.kernel_regularizer)})
            return config

    def transformer_encoder_block(inputs, head_size, num_heads, ff_dim, dropout_rate=0.1, name_prefix="transformer"):
        x = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_norm1")(inputs)
        attn_output = layers.MultiHeadAttention(
            key_dim=head_size, num_heads=num_heads, dropout=dropout_rate,
            name=f"{name_prefix}_mha"
        )(x, x)
        attn_output = layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout1")(attn_output)
        x = layers.Add(name=f"{name_prefix}_add1")([inputs, attn_output])
        
        normed = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_norm2")(x)
        ff_output = layers.Dense(ff_dim, activation='gelu', name=f"{name_prefix}_ff1")(normed)
        ff_output = layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout2")(ff_output)
        ff_output = layers.Dense(inputs.shape[-1], name=f"{name_prefix}_ff2")(ff_output)
        ff_output = layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout3")(ff_output)
        
        return layers.Add(name=f"{name_prefix}_add2")([x, ff_output])

    def build_attention_cnn_extractor(input_shape=(300, 400, 3), reg=0.0005, dropout=0.35):
        img_input = keras.Input(shape=input_shape, name='image_input')
        
        x = layers.Conv2D(16, 7, padding='same', kernel_initializer='he_normal')(img_input)
        x = layers.Conv2D(16, 5, padding='same', kernel_initializer='he_normal')(x)
        local12 = layers.Activation('relu')(x)
        
        x = layers.Conv2D(16, 3, padding='same', kernel_initializer='he_normal')(local12)
        local1 = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(local1)
        x = layers.Dropout(dropout)(x)
        
        x = layers.Conv2D(32, 3, padding='same', kernel_initializer='he_normal')(x)
        local2 = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(local2)
        x = layers.Dropout(dropout)(x)
        
        x = layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal')(x)
        local3 = layers.Activation('relu')(x)
        
        x = layers.Conv2D(128, 3, padding='same', kernel_initializer='he_normal')(local3)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(dropout)(x)
        
        flat = layers.Flatten()(x)
        g = layers.Dense(64, activation='relu')(flat)
        g = layers.Dropout(0.2)(g)
        global_features = layers.Dense(16, activation='relu', name='global_features')(g)
        
        def create_attention_branch(local_feat, branch_name):
            l_proj = layers.Dense(16)(local_feat)
            compatibility = ParametrisedCompatibility(
                kernel_regularizer=regularizers.l2(reg),
                name=f'compatibility_{branch_name}'
            )([l_proj, global_features])
            
            flat_compat = layers.Flatten(name=f'flat_{branch_name}')(compatibility)
            attention_weights = layers.Activation('softmax', name=f'attention_{branch_name}')(flat_compat)
            
            reshaped_local = layers.Reshape((-1, 16), name=f'reshape_{branch_name}')(l_proj)
            attended = layers.Lambda(
                lambda z: K.squeeze(K.batch_dot(K.expand_dims(z[0], 1), z[1]), 1),
                name=f'attended_{branch_name}'
            )([attention_weights, reshaped_local])
            
            return attended
        
        attended1 = create_attention_branch(local1, 'scale1')
        attended2 = create_attention_branch(local2, 'scale2') 
        attended12 = create_attention_branch(local12, 'scale12')
        
        return keras.Model(img_input, [attended1, attended2, attended12, global_features], name='attention_cnn_extractor')

    def build_transformer_extractor(input_shape=(300, 400, 3), patch_size=25, embed_dim=128, 
                                   num_transformer_layers=4, num_heads=8, ff_dim=256, dropout_rate=0.1):
        img_input = keras.Input(shape=input_shape, name='transformer_image_input')
        
        batch_size = tf.shape(img_input)[0]
        height, width = input_shape[0], input_shape[1]
        
        patches = tf.image.extract_patches(
            images=img_input, sizes=[1, patch_size, patch_size, 1],
            strides=[1, patch_size, patch_size, 1], rates=[1, 1, 1, 1], padding='VALID'
        )
        
        patch_dim = patch_size * patch_size * 3
        num_patches = (height // patch_size) * (width // patch_size)
        patches = tf.reshape(patches, [batch_size, num_patches, patch_dim])
        
        patch_embeddings = layers.Dense(embed_dim, name='patch_projection')(patches)
        
        positions = tf.range(start=0, limit=num_patches, delta=1)
        pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=embed_dim, name='pos_embedding')
        pos_encoding = pos_embedding(positions)
        pos_encoding = tf.expand_dims(pos_encoding, axis=0)
        
        x = patch_embeddings + pos_encoding
        
        cls_embedding = layers.Embedding(input_dim=1, output_dim=embed_dim, name="cls_token_embedding")
        cls_input = tf.zeros((batch_size, 1), dtype=tf.int32)
        cls_tokens = cls_embedding(cls_input)
        x = layers.Concatenate(axis=1, name="add_cls_token")([cls_tokens, x])
        
        for i in range(num_transformer_layers):
            x = transformer_encoder_block(
                x, head_size=embed_dim // num_heads, num_heads=num_heads,
                ff_dim=ff_dim, dropout_rate=dropout_rate, name_prefix=f"transformer_layer_{i}"
            )
        
        cls_representation = x[:, 0, :]
        transformer_features = layers.Dense(16, activation='relu', name='transformer_global_features')(cls_representation)
        
        return keras.Model(img_input, transformer_features, name='transformer_extractor')

    def build_combined_multimodal_model(img_shape=(300, 400, 3), num_numeric_feats=1, num_classes=3,
                                       cnn_reg=0.0005, cnn_dropout=0.35, patch_size=25, embed_dim=128,
                                       num_transformer_layers=4, transformer_heads=8, transformer_ff_dim=256,
                                       transformer_dropout=0.15, fusion_dropout=0.3, final_reg=0.001):
        
        image_input = keras.Input(shape=img_shape, name='image_input')
        numeric_input = keras.Input(shape=(num_numeric_feats,), name='numeric_input')
        
        # CNN-Attention branch
        cnn_extractor = build_attention_cnn_extractor(input_shape=img_shape, reg=cnn_reg, dropout=cnn_dropout)
        cnn_features = cnn_extractor(image_input)
        cnn_attended1, cnn_attended2, cnn_attended12, cnn_global = cnn_features
        cnn_combined = layers.Concatenate(name='cnn_attention_fusion')([
            cnn_attended1, cnn_attended2, cnn_attended12, cnn_global
        ])
        
        # Transformer branch
        transformer_extractor = build_transformer_extractor(
            input_shape=img_shape, patch_size=patch_size, embed_dim=embed_dim,
            num_transformer_layers=num_transformer_layers, num_heads=transformer_heads,
            ff_dim=transformer_ff_dim, dropout_rate=transformer_dropout
        )
        transformer_features = transformer_extractor(image_input)
        
        # Numeric feature processing
        numeric_processed = layers.Dense(64, activation='relu', name='numeric_dense1')(numeric_input)
        numeric_processed = layers.Dropout(0.2, name='numeric_dropout1')(numeric_processed)
        numeric_processed = layers.Dense(16, activation='relu', name='numeric_dense2')(numeric_processed)
        
        # Multi-modal fusion
        fused_features = layers.Concatenate(name='multimodal_fusion')([
            cnn_combined, transformer_features, numeric_processed
        ])
        
        # Final classification layers
        x = layers.Dense(128, activation='gelu', 
                        kernel_regularizer=regularizers.l2(final_reg),
                        name='classification_dense1')(fused_features)
        x = layers.Dropout(fusion_dropout, name='classification_dropout1')(x)
        
        x = layers.Dense(64, activation='gelu',
                        kernel_regularizer=regularizers.l2(final_reg),
                        name='classification_dense2')(x)
        x = layers.Dropout(fusion_dropout, name='classification_dropout2')(x)
        
        predictions = layers.Dense(num_classes, activation='softmax',
                                  kernel_regularizer=regularizers.l2(final_reg),
                                  name='predictions')(x)
        
        model = keras.Model([image_input, numeric_input], predictions, name='combined_multimodal_model')
        return model

    # Create and train model for this fold
    print(f"Creating model for fold {fold_num}...")
    fold_model = build_combined_multimodal_model(
        img_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
        num_numeric_feats=1,
        num_classes=len(class_names),
        patch_size=25,
        embed_dim=128,
        num_transformer_layers=4,
        transformer_heads=8
    )
    
    # Compile model
    optimizer = keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4)
    fold_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    # Training callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(f'fold_{fold_num}_best_model.keras', monitor='val_accuracy', save_best_only=True)
    ]
    
    # Train model
    print(f"Training fold {fold_num}...")
    history = fold_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=100,
        callbacks=callbacks,
        verbose=1
    )
    
    # Load best weights and evaluate
    fold_model.load_weights(f'fold_{fold_num}_best_model.keras')
    test_loss, test_acc = fold_model.evaluate(test_dataset, verbose=0)
    
    fold_results.append({
        'fold': fold_num,
        'test_accuracy': test_acc,
        'test_loss': test_loss,
        'best_val_acc': max(history.history['val_accuracy'])
    })
    
    print(f"Fold {fold_num} Results:")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test Loss: {test_loss:.4f}")
    
    # Clean up
    del fold_model
    keras.backend.clear_session()

# Final results
print("\n" + "="*50)
print("5-FOLD CROSS-VALIDATION COMPLETE")
print("="*50)

accuracies = [r['test_accuracy'] for r in fold_results]
print(f"Mean Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")

for r in fold_results:
    print(f"Fold {r['fold']}: Test Acc = {r['test_accuracy']:.4f}")

Starting 5-fold cross-validation with 1578 total samples

==================== FOLD 1/5 ====================
Fold 1 data sizes:
  Train: 1010 samples
  Validation: 252 samples
  Test: 316 samples
Datasets created for fold 1
Creating model for fold 1...
Training fold 1...
Epoch 1/100


2025-09-11 10:22:29.219074: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape incombined_multimodal_model/attention_cnn_extractor/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-09-11 10:22:31.287579: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
2025-09-11 10:22:33.067212: I external/local_xla/xla/service/service.cc:168] XLA service 0x7a064ce2dbe0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-11 10:22:33.067288: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2025-09-11 10:22:33.082240: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1757586153.328522    1287 device_comp

32/32 [==============================] - 50s 379ms/step - loss: 1.9866 - accuracy: 0.3129 - val_loss: 1.2962 - val_accuracy: 0.2937 - lr: 2.0000e-04
Epoch 2/100
32/32 [==============================] - 5s 139ms/step - loss: 1.2955 - accuracy: 0.3762 - val_loss: 1.2841 - val_accuracy: 0.6706 - lr: 2.0000e-04
Epoch 3/100
32/32 [==============================] - 4s 102ms/step - loss: 1.2789 - accuracy: 0.4406 - val_loss: 1.2732 - val_accuracy: 0.4603 - lr: 2.0000e-04
Epoch 4/100
32/32 [==============================] - 5s 139ms/step - loss: 1.2516 - accuracy: 0.5416 - val_loss: 1.2373 - val_accuracy: 0.6865 - lr: 2.0000e-04
Epoch 5/100
32/32 [==============================] - 4s 116ms/step - loss: 1.1773 - accuracy: 0.5782 - val_loss: 1.1807 - val_accuracy: 0.4722 - lr: 2.0000e-04
Epoch 6/100
32/32 [==============================] - 5s 145ms/step - loss: 1.0457 - accuracy: 0.6297 - val_loss: 0.9108 - val_accuracy: 0.7778 - lr: 2.0000e-04
Epoch 7/100
32/32 [==============================] 

2025-09-11 10:30:30.379251: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape incombined_multimodal_model/attention_cnn_extractor/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


32/32 [==============================] - 33s 184ms/step - loss: 2.3535 - accuracy: 0.3317 - val_loss: 1.3015 - val_accuracy: 0.3135 - lr: 2.0000e-04
Epoch 2/100
32/32 [==============================] - 5s 140ms/step - loss: 1.3013 - accuracy: 0.3644 - val_loss: 1.2862 - val_accuracy: 0.4722 - lr: 2.0000e-04
Epoch 3/100
32/32 [==============================] - 5s 142ms/step - loss: 1.2885 - accuracy: 0.3743 - val_loss: 1.2682 - val_accuracy: 0.6349 - lr: 2.0000e-04
Epoch 4/100
32/32 [==============================] - 4s 104ms/step - loss: 1.2623 - accuracy: 0.4663 - val_loss: 1.2403 - val_accuracy: 0.6349 - lr: 2.0000e-04
Epoch 5/100
32/32 [==============================] - 5s 139ms/step - loss: 1.2244 - accuracy: 0.5614 - val_loss: 1.1905 - val_accuracy: 0.6389 - lr: 2.0000e-04
Epoch 6/100
32/32 [==============================] - 5s 140ms/step - loss: 1.1562 - accuracy: 0.6040 - val_loss: 1.0945 - val_accuracy: 0.6667 - lr: 2.0000e-04
Epoch 7/100
32/32 [==============================] 

...

32/32 [==============================] - 32s 182ms/step - loss: 2.9943 - accuracy: 0.3634 - val_loss: 1.3023 - val_accuracy: 0.3135 - lr: 2.0000e-04
Epoch 2/100
32/32 [==============================] - 5s 145ms/step - loss: 1.2899 - accuracy: 0.3911 - val_loss: 1.2797 - val_accuracy: 0.3254 - lr: 2.0000e-04
Epoch 3/100
32/32 [==============================] - 4s 104ms/step - loss: 1.2714 - accuracy: 0.4297 - val_loss: 1.2607 - val_accuracy: 0.3254 - lr: 2.0000e-04
Epoch 4/100
32/32 [==============================] - 4s 101ms/step - loss: 1.2453 - accuracy: 0.4683 - val_loss: 1.2313 - val_accuracy: 0.3254 - lr: 2.0000e-04
Epoch 5/100
32/32 [==============================] - 5s 150ms/step - loss: 1.2025 - accuracy: 0.5802 - val_loss: 1.1501 - val_accuracy: 0.7143 - lr: 2.0000e-04
Epoch 6/100
32/32 [==============================] - 4s 107ms/step - loss: 1.1302 - accuracy: 0.6416 - val_loss: 1.0340 - val_accuracy: 0.7024 - lr: 2.0000e-04
Epoch 7/100
32/32 [==============================] 

2025-09-11 10:41:28.113012: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape incombined_multimodal_model/attention_cnn_extractor/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


32/32 [==============================] - 36s 327ms/step - loss: 4.1600 - accuracy: 0.3284 - val_loss: 1.2884 - val_accuracy: 0.3571 - lr: 2.0000e-04
Epoch 2/100
32/32 [==============================] - 5s 144ms/step - loss: 1.3139 - accuracy: 0.3561 - val_loss: 1.2689 - val_accuracy: 0.7500 - lr: 2.0000e-04
Epoch 3/100
32/32 [==============================] - 4s 100ms/step - loss: 1.2668 - accuracy: 0.4402 - val_loss: 1.2439 - val_accuracy: 0.6429 - lr: 2.0000e-04
Epoch 4/100
32/32 [==============================] - 4s 103ms/step - loss: 1.2439 - accuracy: 0.5232 - val_loss: 1.2041 - val_accuracy: 0.6944 - lr: 2.0000e-04
Epoch 5/100
32/32 [==============================] - 4s 103ms/step - loss: 1.1899 - accuracy: 0.6281 - val_loss: 1.1440 - val_accuracy: 0.7460 - lr: 2.0000e-04
Epoch 6/100
32/32 [==============================] - 4s 102ms/step - loss: 1.1161 - accuracy: 0.6469 - val_loss: 1.0475 - val_accuracy: 0.7381 - lr: 2.0000e-04
Epoch 7/100
32/32 [==============================] 

2025-09-11 10:48:46.513637: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape incombined_multimodal_model/attention_cnn_extractor/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


32/32 [==============================] - 34s 194ms/step - loss: 2.1493 - accuracy: 0.3304 - val_loss: 1.2897 - val_accuracy: 0.4365 - lr: 2.0000e-04
Epoch 2/100
32/32 [==============================] - 4s 112ms/step - loss: 1.3241 - accuracy: 0.3393 - val_loss: 1.2800 - val_accuracy: 0.3413 - lr: 2.0000e-04
Epoch 3/100
32/32 [==============================] - 4s 109ms/step - loss: 1.2747 - accuracy: 0.4362 - val_loss: 1.2608 - val_accuracy: 0.3413 - lr: 2.0000e-04
Epoch 4/100
32/32 [==============================] - 4s 110ms/step - loss: 1.2486 - accuracy: 0.4738 - val_loss: 1.2277 - val_accuracy: 0.3413 - lr: 2.0000e-04
Epoch 5/100
32/32 [==============================] - 6s 172ms/step - loss: 1.2130 - accuracy: 0.5460 - val_loss: 1.1760 - val_accuracy: 0.7738 - lr: 2.0000e-04
Epoch 6/100
32/32 [==============================] - 5s 149ms/step - loss: 1.1551 - accuracy: 0.5856 - val_loss: 1.0931 - val_accuracy: 0.8532 - lr: 2.0000e-04
Epoch 7/100
32/32 [==============================] 

In [32]:
# Add this line after your existing results code
test_losses = [r['test_loss'] for r in fold_results]
print(f"Mean Test Loss: {np.mean(test_losses):.4f} ± {np.std(test_losses):.4f}")

# You can also print individual fold losses
print("\nIndividual fold results:")
for r in fold_results:
    print(f"Fold {r['fold']}: Test Acc = {r['test_accuracy']:.4f}, Test Loss = {r['test_loss']:.4f}")

Mean Test Loss: 0.4207 ± 0.0245

Individual fold results:
Fold 1: Test Acc = 0.8829, Test Loss = 0.4454
Fold 2: Test Acc = 0.8892, Test Loss = 0.4467
Fold 3: Test Acc = 0.9272, Test Loss = 0.3887
Fold 4: Test Acc = 0.9048, Test Loss = 0.4276
Fold 5: Test Acc = 0.9175, Test Loss = 0.3951
